# PlantHealth AI – Custom Convolutional Neural Network (From Scratch)

This notebook demonstrates building, training, and evaluating a **Custom Deep CNN** architecture from scratch for 38-class plant disease classification on the PlantVillage dataset.

### Pipeline Stages:
1. Environment and Dependency Verification
2. Dataset Discovery and Anti-Leakage Stratified Splitting
3. Training Data Augmentation & Normalization Pipeline
4. CNN Architecture Construction (Conv2D, BatchNorm, Pooling, Dropout)
5. Model Compilation and Checkpoint Callbacks
6. Model Training and Convergence Curves
7. Quantitative Evaluation on Untouched Test Split
8. Confusion Matrix & Qualitative Predictions

In [1]:
import os
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow import keras

from config import DATA_DIR, NUM_CLASSES, INPUT_SHAPE, BATCH_SIZE, CUSTOM_CNN_MODEL_PATH
from src.data_loader import discover_dataset, build_stratified_dataset_splits, create_tf_dataset
from src.custom_cnn import build_custom_cnn
from src.augmentation import get_training_augmentation_pipeline, visualize_augmentations
from src.utils import plot_training_history, plot_confusion_matrix, plot_prediction_samples

print(f"TensorFlow Version: {tf.__version__}")
print(f"Available Devices: {tf.config.list_physical_devices()}")

TensorFlow Version: 2.21.0
Available Devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


## 2. Dataset Discovery and Stratified Partitioning
We discover the 38 classes and split the dataset with strict anti-leakage grouping.

In [2]:
all_items, class_to_idx, idx_to_class = discover_dataset(DATA_DIR)
class_names = [idx_to_class[i] for i in range(len(idx_to_class))]

# Stratified Anti-Leakage Split (70% Train, 15% Val, 15% Test)
train_samples, val_samples, test_samples = build_stratified_dataset_splits(
    all_items, class_to_idx, max_samples_per_class=60
)
print(f"Train samples: {len(train_samples)}, Val samples: {len(val_samples)}, Test samples: {len(test_samples)}")

[2026-10-31 21:51:50] [INFO] PlantHealthAI: Saved JSON artifact to /Users/manthanshah/Documents/PlantHealth-AI/models/class_indices.json


[2026-10-31 21:51:50] [INFO] PlantHealthAI: Discovered 87867 valid images across 38 classes.


[2026-10-31 21:51:50] [INFO] PlantHealthAI: Stratified Splits Built (No Leakage): Train=6990 (77.8%), Val=993 (11.0%), Test=1006 (11.2%)


Train samples: 6990, Val samples: 993, Test samples: 1006


## 3. Augmentation Visualization
We inspect real-time data augmentations applied strictly to the training split.

In [3]:
sample_path, sample_label = train_samples[0]
sample_img = tf.keras.utils.load_img(sample_path, target_size=(224, 224))
sample_arr = tf.keras.utils.img_to_array(sample_img)

aug_pipe = get_training_augmentation_pipeline()
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
axes[0].imshow(sample_arr.astype("uint8"))
axes[0].set_title("Original")
axes[0].axis("off")

for i in range(1, 5):
    augmented = aug_pipe(np.expand_dims(sample_arr, 0), training=True)
    axes[i].imshow(np.squeeze(augmented.numpy()).astype("uint8"))
    axes[i].set_title(f"Augmented #{i}")
    axes[i].axis("off")
plt.tight_layout()
plt.show()

/var/folders/yd/smv0lx896fv72dkkr_crbntw0000gn/T/ipykernel_36795/127540994.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Custom CNN Architecture Construction
Instantiate the 4-block deep CNN with Batch Normalization, Max-Pooling, Dropout, and Global Average Pooling.

In [4]:
model = build_custom_cnn(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES)
model.summary()

Model: "PlantHealth_CustomCNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_image (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1_1 (Conv2D)                │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn1_1 (BatchNormalization)      │ (None, 224, 224, 32)   │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu1_1 (Activation)            │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1_2 (Conv2D)                │ (None, 224, 224, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn1_2 (BatchNormalization)      │ (None, 224, 224, 32)   │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu1_2 (Activation)            │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool1 (MaxPooling2D)            │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop1 (Dropout)                 │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2_1 (Conv2D)                │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn2_1 (BatchNormalization)      │ (None, 112, 112, 64)   │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu2_1 (Activation)            │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2_2 (Conv2D)                │ (None, 112, 112, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn2_2 (BatchNormalization)      │ (None, 112, 112, 64)   │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu2_2 (Activation)            │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool2 (MaxPooling2D)            │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop2 (Dropout)                 │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3_1 (Conv2D)                │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn3_1 (BatchNormalization)      │ (None, 56, 56, 128)    │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu3_1 (Activation)            │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3_2 (Conv2D)                │ (None, 56, 56, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn3_2 (BatchNormalization)      │ (None, 56, 56, 128)    │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu3_2 (Activation)            │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool3 (MaxPooling2D)            │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop3 (Dropout)                 │ (None, 28, 28, 128)    │             

 Total params: 661,574 (2.52 MB)

 Trainable params: 659,654 (2.52 MB)

 Non-trainable params: 1,920 (7.50 KB)

## 5. Model Training or Checkpoint Loading
We check for existing trained weights at `models/custom_cnn.keras` or perform training with callbacks.

In [5]:
train_ds = create_tf_dataset(train_samples, batch_size=32, model_type="custom_cnn", is_training=True)
val_ds = create_tf_dataset(val_samples, batch_size=32, model_type="custom_cnn", is_training=False)
test_ds = create_tf_dataset(test_samples, batch_size=32, model_type="custom_cnn", is_training=False)

if CUSTOM_CNN_MODEL_PATH.exists():
    print(f"Loading verified trained checkpoint from {CUSTOM_CNN_MODEL_PATH}...")
    model = keras.models.load_model(CUSTOM_CNN_MODEL_PATH)
    print("Model successfully loaded.")
else:
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True),
        keras.callbacks.ModelCheckpoint(str(CUSTOM_CNN_MODEL_PATH), save_best_only=True)
    ]
    history = model.fit(train_ds, validation_data=val_ds, epochs=1, callbacks=callbacks)

Loading verified trained checkpoint from /Users/manthanshah/Documents/PlantHealth-AI/models/custom_cnn.keras...
Model successfully loaded.


## 6. Evaluation on Untouched Test Set
Assess generalization accuracy, loss, and confusion matrix.

In [6]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"Custom CNN Test Accuracy: {test_acc:.2%}")
print(f"Custom CNN Test Loss: {test_loss:.4f}")

 1/32 ━━━━━━━━━━━━━━━━━━━━ 14s 472ms/step - accuracy: 0.0000e+00 - loss: 4.2725

 2/32 ━━━━━━━━━━━━━━━━━━━━ 8s 272ms/step - accuracy: 0.0000e+00 - loss: 4.2516 

 3/32 ━━━━━━━━━━━━━━━━━━━━ 7s 271ms/step - accuracy: 0.0000e+00 - loss: 4.2598

 4/32 ━━━━━━━━━━━━━━━━━━━━ 7s 271ms/step - accuracy: 0.0000e+00 - loss: 4.2480

 5/32 ━━━━━━━━━━━━━━━━━━━━ 7s 269ms/step - accuracy: 0.0000e+00 - loss: 4.2174

 6/32 ━━━━━━━━━━━━━━━━━━━━ 7s 269ms/step - accuracy: 0.0000e+00 - loss: 4.1862

... (24 outputs left out)

31/32 ━━━━━━━━━━━━━━━━━━━━ 0s 285ms/step - accuracy: 0.0000e+00 - loss: 4.2076

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 280ms/step - accuracy: 0.0000e+00 - loss: 4.2037

32/32 ━━━━━━━━━━━━━━━━━━━━ 9s 280ms/step - accuracy: 0.0000e+00 - loss: 4.2037


Custom CNN Test Accuracy: 0.00%
Custom CNN Test Loss: 4.2037
